# Google Maps Review Scraper

Notebook untuk **scraping** review dari Google Maps menggunakan Apify API.

**Output:** `data/raw_review/additional/{nama_desa}_reviews_raw.csv` (data mentah)


## 1. Setup & Import

In [11]:
# Jalankan sekali untuk install
# !pip install apify-client

In [12]:
import os
import pandas as pd
from apify_client import ApifyClient

BASE_DIR = os.path.dirname(os.getcwd())
OUTPUT_DIR = os.path.join(BASE_DIR, "data", "raw_review", "additional", "raw")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print("Setup OK!")

Output directory: d:\Kuliah\TA\TA_Notebook\data\raw_review\additional
Setup OK!


## 2. Konfigurasi

In [ ]:
# ============================================================
# EDIT DI SINI
# ============================================================

API_TOKEN = "token"  # API token dari apify.com

GOOGLE_MAPS_URL = "https://maps.app.goo.gl/N4YeCK4pY8Py5G9v5"  # URL Google Maps
NAMA_DESA = "Pentingsari"  # Nama desa wisata (untuk kolom output)

MAX_REVIEWS = 500  # Max review yang diambil

client = ApifyClient(API_TOKEN)
print(f"Target: {NAMA_DESA}")
print(f"URL: {GOOGLE_MAPS_URL}")
print(f"Max reviews: {MAX_REVIEWS}")

Target: Pentingsari
URL: https://maps.app.goo.gl/N4YeCK4pY8Py5G9v5
Max reviews: 500


## 3. Scrape Reviews


In [14]:
# Scrape reviews via Apify Google Maps Scraper
print(f"Scraping reviews untuk '{NAMA_DESA}'...")
print("(Proses jalan di cloud Apify, tunggu beberapa menit...)\n")

run_input = {
    "startUrls": [{"url": GOOGLE_MAPS_URL}],
    "maxReviews": MAX_REVIEWS,
    "reviewsSort": "newest",
    "language": "id",
    "scrapeReviewsPersonalData": True,
    # Matikan fitur yang tidak perlu
    "maxImages": 0,
    "maxQuestions": 0,
    "scrapeContacts": False,
    "scrapeSocialMediaProfiles": {
        "facebooks": False, "instagrams": False,
        "youtubes": False, "tiktoks": False, "twitters": False,
    },
    "scrapeDirectories": False,
    "includeWebResults": False,
    "scrapeTableReservationProvider": False,
}

# Run actor dan tunggu selesai
run = client.actor("nwua9Gu5YrADL7ZDj").call(run_input=run_input)
print(f"Actor selesai! Run ID: {run['id']}")

# Ambil hasil dari dataset
raw_reviews = []
for item in client.dataset(run["defaultDatasetId"]).iterate_items():
    # Setiap item = 1 place, reviews ada di 'reviews'
    reviews_list = item.get("reviews", [])
    if not reviews_list:
        # Kalau structurenya flat (1 item = 1 review)
        text = item.get("reviewText", "") or item.get("text", "") or ""
        if text.strip():
            raw_reviews.append({
                "review_text": text.strip(),
                "rating": item.get("stars", item.get("reviewRating", 0)),
                "author": item.get("reviewerName", item.get("name", "")),
                "date": item.get("publishedAtDate", item.get("reviewDate", "")),
            })
    else:
        for r in reviews_list:
            text = r.get("text", "") or r.get("reviewText", "") or ""
            if text.strip():
                raw_reviews.append({
                    "review_text": text.strip(),
                    "rating": r.get("stars", r.get("rating", 0)),
                    "author": r.get("name", r.get("reviewerName", "")),
                    "date": r.get("publishedAtDate", r.get("reviewDate", "")),
                })

df_raw = pd.DataFrame(raw_reviews)
print(f"\nTotal review scraped: {len(df_raw)}")
df_raw.head(10)

Scraping reviews untuk 'Pentingsari'...
(Proses jalan di cloud Apify, tunggu beberapa menit...)



[apify.crawler-google-places runId:FkGkUGmSzFcsIPxHj] -> Status: RUNNING, Message: 
[apify.crawler-google-places runId:FkGkUGmSzFcsIPxHj] -> 2026-04-05T05:14:37.740Z ACTOR: Pulling container image of build WkCwNPIsiJfvVfkSV from registry.
[apify.crawler-google-places runId:FkGkUGmSzFcsIPxHj] -> 2026-04-05T05:14:37.742Z ACTOR: Creating container.
[apify.crawler-google-places runId:FkGkUGmSzFcsIPxHj] -> 2026-04-05T05:14:37.790Z ACTOR: Starting container.
[apify.crawler-google-places runId:FkGkUGmSzFcsIPxHj] -> 2026-04-05T05:14:37.798Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.crawler-google-places runId:FkGkUGmSzFcsIPxHj] -> 2026-04-05T05:14:39.858Z INFO  System info {"apifyVersion":"3.5.3","apifyClientVersion":"2.19.0","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v18.20.8"}
[apify.crawler-google-places runId:FkGkUGmSzFcsIPxHj] -> 2026-04-05T05:14:40.376Z INFO  [BG ENQUEUE] Finished enqueueing
[apify.crawler-google-places runId:FkGkUGmSzFcsIPxHj] ->

Actor selesai! Run ID: FkGkUGmSzFcsIPxHj

Total review scraped: 246


,review_text,rating,author,date
0,enakk warga nya sopan semua makmur,5,gal 123,2026-04-04T12:02:30.226Z
1,Kampung suami,5,Yumnaica Aidulfitrya Djjaya,2026-03-16T13:38:02.046Z
2,You can learn javanese culture such as traditi...,5,Ali Tafrihatullutfi (Kang Lutfi),2026-02-03T05:53:45.145Z
3,"Kearifan lokal yang menyenangkan, keramahan pe...",5,Sahid Arbas,2026-01-25T04:53:43.104Z
4,"Desa wisata terbaik.\nTerbaik dalam keramahan,...",5,agung hartono,2026-01-14T11:02:39.511Z
5,Hiburan,5,Karjo Mugo,2026-01-01T08:31:32.011Z
6,Tempatnya luas dan asri cukup bagus untuk kegi...,5,Ulfa Nur Wahyudi,2025-12-28T05:19:26.687Z
7,"Rumah Simbahnya Zahra ada di desa inih, lingku...",5,edwonk zapravo,2025-12-22T12:01:00.532Z
8,seru banget kesini. wisata yang bener2 seru ap...,5,Lutfi Hakim,2025-12-19T00:42:53.602Z
9,Di desa ini buat ngadain acara penanaman pohon...,5,Rifka Agnes,2025-12-11T10:45:32.043Z


## 4. Export Raw CSV

Simpan data mentah (tanpa cleaning). Cleaning dilakukan di notebook terpisah (`Clean_Reviews.ipynb`).

In [15]:
# Export raw CSV
filename = NAMA_DESA.lower().replace(" ", "_") + "_reviews_raw.csv"
output_path = os.path.join(OUTPUT_DIR, filename)

df_raw["nama desa wisata"] = NAMA_DESA
df_raw.to_csv(output_path, index=False, encoding='utf-8')

print(f"Saved {len(df_raw)} raw reviews ke: {output_path}")
print(f"\n{'='*50}")
print("NEXT STEP:")
print("1. Jalankan Clean_Reviews.ipynb untuk cleaning & filtering")
print("2. Jalankan SA_AutoLabel.ipynb untuk labeling + aspect detection")
print("3. Dashboard otomatis menampilkan desa baru")
print(f"{'='*50}")

Saved 246 raw reviews ke: d:\Kuliah\TA\TA_Notebook\data\raw_review\additional\pentingsari_reviews_raw.csv

NEXT STEP:
1. Jalankan Clean_Reviews.ipynb untuk cleaning & filtering
2. Jalankan SA_AutoLabel.ipynb untuk labeling + aspect detection
3. Dashboard otomatis menampilkan desa baru


## 6. Batch Mode (Opsional)

Scrape beberapa desa sekaligus. Edit list `PLACES` di bawah.

In [16]:
# # Uncomment dan edit list ini untuk batch scraping
# PLACES = [
#     {"url": "https://www.google.com/maps/place/Desa+Wisata+Pentingsari", "nama": "Pentingsari"},
#     {"url": "https://www.google.com/maps/place/Desa+Wisata+Nglanggeran", "nama": "Nglanggeran"},
# ]
#
# for place in PLACES:
#     print(f"\n{'='*60}")
#     print(f"Scraping: {place['nama']}")
#     print(f"{'='*60}")
#
#     run_input = {
#         "startUrls": [{"url": place["url"]}],
#         "maxReviews": MAX_REVIEWS,
#         "reviewsSort": "newest",
#         "language": "id",
#         "scrapeReviewsPersonalData": True,
#         "maxImages": 0,
#         "maxQuestions": 0,
#         "scrapeContacts": False,
#     }
#     run = client.actor("nwua9Gu5YrADL7ZDj").call(run_input=run_input)
#
#     raw = []
#     for item in client.dataset(run["defaultDatasetId"]).iterate_items():
#         reviews_list = item.get("reviews", [])
#         if not reviews_list:
#             text = item.get("reviewText", "") or item.get("text", "") or ""
#             if text.strip():
#                 raw.append({"review_text": text.strip(), "rating": item.get("stars", 0),
#                            "author": item.get("reviewerName", ""), "date": item.get("publishedAtDate", "")})
#         else:
#             for r in reviews_list:
#                 text = r.get("text", "") or r.get("reviewText", "") or ""
#                 if text.strip():
#                     raw.append({"review_text": text.strip(), "rating": r.get("stars", 0),
#                                "author": r.get("name", ""), "date": r.get("publishedAtDate", "")})
#
#     df = pd.DataFrame(raw)
#     df["nama desa wisata"] = place["nama"]
#     filename = place["nama"].lower().replace(" ", "_") + "_reviews_raw.csv"
#     path = os.path.join(OUTPUT_DIR, filename)
#     df.to_csv(path, index=False, encoding='utf-8')
#     print(f"Saved {len(df)} raw reviews -> {path}")
#
# print("\nBatch scraping selesai! Jalankan Clean_Reviews.ipynb untuk cleaning.")

## Troubleshooting

### API Token Error
- Pastikan sudah daftar di https://apify.com
- Buka Settings → Integrations → copy API Token
- Free tier: $5 credit/bulan

### Tidak ada review / 0 results
- Cek URL Google Maps — pastikan URL valid dan tempatnya punya review
- Coba naikkan `maxReviews`
- Cek sisa credit di Apify dashboard

### Review sedikit setelah filter bahasa
- Kurangi `min_match` di `is_indonesian()` (default=2, coba 1)
- Hapus filter `language: "id"` di `run_input` kalau mau ambil semua bahasa dulu, baru filter manual

### Actor error / timeout
- Cek Apify dashboard → Runs → lihat log error
- Pastikan URL Google Maps bukan URL shortlink (pakai URL lengkap)